# Kaggle の独自データセットで行う意味的セグメンテーション

この Notebook は、画像ディレクトリとマスクディレクトリを指定して U-Net を学習するための例です。各セルには 1 つの責務だけを置いています。クラス数は全マスクから自動検出するため、既定値を誤って使いません。

## 1. リポジトリを準備する

In [ ]:
%cd /kaggle/working
![ -d /kaggle/working/torch-foundry/.git ] || git clone https://github.com/Kaz0818/torch-foundry.git torch-foundry
%cd /kaggle/working/torch-foundry
!pip install -e .

## 2. 必要な機能を読み込む

In [ ]:
import json
import random
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import wandb
from torch import optim

from vision.segmentation.config import Config
from vision.segmentation.datasets.dataset import get_dataloader
from vision.segmentation.datasets.paths import (
    detect_mask_class_ids,
    pair_image_mask_paths,
)
from vision.segmentation.losses import build_loss
from vision.segmentation.models.unet import UNet
from vision.segmentation.training.train import train
from vision.segmentation.utils.visualization import get_device, plot_overlay

PROJECT_ROOT = Path.cwd()

## 3. 実験設定を入力する

このセルだけを編集します。画像・マスクのパスと学習条件を設定してください。クラス数は次のセルで自動検出します。

In [ ]:
DATASET_NAME = "NPPE1 semantic segmentation"
IMAGE_DIRECTORY = Path(
    "/kaggle/input/competitions/nppe-1-dl-gen-ai-course-semantic-segmentation/"
    "Dataset/train/images"
)
MASK_DIRECTORY = Path(
    "/kaggle/input/competitions/nppe-1-dl-gen-ai-course-semantic-segmentation/"
    "Dataset/train/masks"
)
OUTPUT_ROOT = Path("/kaggle/working/segmentation-runs")

IMAGE_SIZE = (256, 256)
BATCH_SIZE = 8
VALIDATION_RATIO = 0.2
SEED = 42
NUM_EPOCHS = 30
LEARNING_RATE = 1e-3
LOSS_NAME = "ce_dice"  # "cross_entropy" または "ce_dice"
CE_WEIGHT = 0.5
DICE_WEIGHT = 0.5
DICE_INCLUDE_BACKGROUND = False

ENABLE_WANDB = True
WANDB_SECRET_NAME = "WANDB_API_KEY"
WANDB_PROJECT = "NPPE1-DLGen_segmentation"
WANDB_RUN_NAME = "baseline-ce-dice"

RUN_CPU_PREFLIGHT = True

## 4. W&B に認証する

W&B を使わない場合は、前のセルの `ENABLE_WANDB` を `False` にしてください。

In [ ]:
if ENABLE_WANDB:
    from kaggle_secrets import UserSecretsClient

    if not WANDB_PROJECT:
        raise ValueError("WANDB_PROJECT を設定してください")
    api_key = UserSecretsClient().get_secret(WANDB_SECRET_NAME)
    wandb.login(key=api_key)

## 5. データセットを検証してクラス数を決める

画像とマスクは拡張子を除いたファイル名で対応付けます。全マスクのクラス ID が `0` から連続していることを確認し、検出結果からクラス数を決めます。

In [ ]:
images, masks = pair_image_mask_paths(IMAGE_DIRECTORY, MASK_DIRECTORY)
class_ids = detect_mask_class_ids(masks)
num_classes = len(class_ids)

print(f"paired samples: {len(images)}")
print(f"detected class IDs: {list(class_ids)}")
print(f"num_classes: {num_classes}")

## 6. 学習設定と DataLoader を作る

In [ ]:
config = Config(
    num_classes=num_classes,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    val_ratio=VALIDATION_RATIO,
    seed=SEED,
    num_epochs=NUM_EPOCHS,
    learning_rate=LEARNING_RATE,
    wandb_enabled=ENABLE_WANDB,
    loss_name=LOSS_NAME,
    ce_weight=CE_WEIGHT,
    dice_weight=DICE_WEIGHT,
    dice_include_background=DICE_INCLUDE_BACKGROUND,
)

train_loader, val_loader = get_dataloader(
    images=images,
    masks=masks,
    batch_size=config.batch_size,
    image_size=config.image_size,
    val_ratio=config.val_ratio,
    seed=config.seed,
)

print(config)
print(f"train samples: {len(train_loader.dataset)}")
print(f"validation samples: {len(val_loader.dataset)}")

## 7. CPU で 1 バッチを検証する

GPU 学習前に、マスクの形状・クラス ID・損失関数・逆伝播を CPU で検証します。CUDA の device-side assert を起こす設定不一致を早期に検出できます。

In [ ]:
if RUN_CPU_PREFLIGHT:
    batch_images, batch_masks = next(iter(train_loader))
    batch_images = batch_images[:1]
    batch_masks = batch_masks[:1]

    assert batch_masks.ndim == 3
    assert int(batch_masks.min()) >= 0
    assert int(batch_masks.max()) < config.num_classes

    cpu_model = UNet(3, config.num_classes)
    cpu_logits = cpu_model(batch_images)
    assert cpu_logits.shape == (
        batch_images.size(0),
        config.num_classes,
        *config.image_size,
    )

    cpu_loss = build_loss(config)(cpu_logits, batch_masks)
    cpu_loss.backward()
    assert torch.isfinite(cpu_loss)
    print(f"CPU preflight loss: {float(cpu_loss):.6f}")

## 8. GPU の学習部品と保存先を準備する

In [ ]:
random.seed(config.seed)
np.random.seed(config.seed)
torch.manual_seed(config.seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(config.seed)

device = get_device()
model = UNet(3, config.num_classes).to(device)
criterion = build_loss(config)
optimizer = optim.Adam(model.parameters(), lr=config.learning_rate)
total_parameters = sum(parameter.numel() for parameter in model.parameters())

output_dir = OUTPUT_ROOT / datetime.now().astimezone().strftime(
    "%Y%m%d_%H%M%S_%f"
)
output_dir.mkdir(parents=True, exist_ok=False)

run_config = {
    "training_config": config.to_dict(),
    "dataset": {
        "name": DATASET_NAME,
        "image_directory": str(IMAGE_DIRECTORY),
        "mask_directory": str(MASK_DIRECTORY),
        "detected_class_ids": list(class_ids),
        "train_samples": len(train_loader.dataset),
        "validation_samples": len(val_loader.dataset),
    },
    "model": {
        "name": "UNet",
        "total_parameters": total_parameters,
    },
    "device": str(device),
}
(output_dir / "config.json").write_text(
    json.dumps(run_config, indent=2) + "\n",
    encoding="utf-8",
)

print(f"device: {device}")
print(f"output directory: {output_dir}")
print(f"total parameters: {total_parameters}")

## 9. 学習を実行して W&B へ記録する

In [ ]:
wandb_mode = None if ENABLE_WANDB else "disabled"
run = wandb.init(
    project=WANDB_PROJECT if ENABLE_WANDB else None,
    name=WANDB_RUN_NAME if ENABLE_WANDB else None,
    config=run_config,
    dir=str(output_dir),
    mode=wandb_mode,
)

if ENABLE_WANDB:
    run.define_metric("epoch")
    run.define_metric("train/*", step_metric="epoch")
    run.define_metric("val/*", step_metric="epoch")

history = train(
    train_loader=train_loader,
    val_loader=val_loader,
    model=model,
    criterion=criterion,
    optimizer=optimizer,
    device=device,
    num_epochs=config.num_epochs,
    num_classes=config.num_classes,
    output_dir=output_dir,
    metric_logger=run.log if ENABLE_WANDB else None,
)

## 10. 検証予測を確認して実行を終了する

In [ ]:
val_images, val_masks = next(iter(val_loader))
model.eval()
with torch.inference_mode():
    val_logits = model(val_images.to(device))
    val_predictions = val_logits.argmax(dim=1).cpu()

comparison = plot_overlay(
    val_images,
    val_masks,
    val_predictions,
    max_images=5,
)
try:
    if ENABLE_WANDB:
        run.log({"results/predictions": wandb.Image(comparison)})
    plt.show()
finally:
    plt.close(comparison)
    run.finish()

print(f"saved epochs: {len(history['train_loss'])}")
print(f"local artifacts: {output_dir}")